# Per-Patient Log-Normalised Objects 

**Goal.** Write one log-normalised `.h5ad` per patient sample, for per-patient CellPhoneDB runs.

**Input.** `new_for_Cellphone_cell_state_fixed.h5ad` (notebook C00): full atlas, raw counts in
`layers['raw_counts']`, cell states in `cell_state_x`.

**Steps.** Remove liver metastases → for each patient sample in each treatment group: subset →
log-normalise from raw counts (counts per 10k, log1p) → save.

**Output.** `cellphoneDB_adata_per_patient/<patient>_log_normalized.h5ad`
(`X` = log-normalised, `layers['raw_counts']` = raw counts, cell state in `cell_state_x`).

In [ ]:
# !pip install --quiet scanpy
# %cd <project>/cpdb

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc

## Config

In [ ]:
input_path = "new_for_Cellphone_cell_state_fixed.h5ad"

output_dir = "cellphoneDB_adata_per_patient"

label_col = "cell_state_x"
patient_col = "ID_2"                    # one value per patient sample (adjacent normal = <ID>_AN)
raw_layer = "raw_counts"
target_sum = 1e4

exclude_patients = {"R008": "liver metastasis", "R011": "liver metastasis"}

# treatment group -> patient samples to export
groups = {
    "AN (adjacent normal)":    [f"TN00{i}_AN" for i in range(1, 10)],
    "TN (treatment naive)":    [f"TN00{i}" for i in range(1, 10)],
    "TN-D (treatment naive D)": ["D001", "D002", "D003"],
    "OT (on treatment)":       ["P001", "P002", "P003", "P005", "P006", "P008", "P009", "P011", "P012", "P013"],
    "TKIR (TKI resistance)":   ["R001", "R003", "R005", "R006", "R007", "R009", "R010"],
}

os.makedirs(output_dir, exist_ok=True)

## 1. Load and remove excluded patients

In [ ]:
adata = sc.read_h5ad(input_path)
vals = adata.layers[raw_layer][:1000].data
assert np.allclose(vals, np.round(vals)), f"'{raw_layer}' must contain raw counts"

adata = adata[~adata.obs["ID"].isin(exclude_patients)]
patients = [p for ps in groups.values() for p in ps]
missing = sorted(set(patients) - set(adata.obs[patient_col]))
assert not missing, f"patients not found: {missing}"

## 2. Log-normalise and save per patient

Normalisation starts from raw counts. `X` in the atlas is already log-transformed, so normalising
`X` again (as before) would log-transform twice.

In [ ]:
summary = []
for group, ids in groups.items():
    for patient in ids:
        keep = (adata.obs[patient_col] == patient).to_numpy()
        a = sc.AnnData(adata.layers[raw_layer][keep].copy(), obs=adata.obs.loc[keep].copy(),
                       var=adata.var[[]].copy())
        a.layers["raw_counts"] = a.X.copy()
        sc.pp.normalize_total(a, target_sum=target_sum)
        sc.pp.log1p(a)
        a.write_h5ad(f"{output_dir}/{patient}_log_normalized.h5ad")
        summary.append({"group": group, "patient": patient, "cells": a.n_obs,
                        "cell_states": a.obs[label_col].nunique()})

summary = pd.DataFrame(summary).set_index(["group", "patient"])
summary.to_csv(f"{output_dir}/summary_cells_per_patient.csv")
summary

In [ ]:
# cells per state and patient (states with few cells in a patient give unstable CellPhoneDB results)
pd.crosstab(adata.obs.loc[adata.obs[patient_col].isin(patients), label_col],
            adata.obs.loc[adata.obs[patient_col].isin(patients), patient_col]).reindex(columns=patients)